[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/OlivierGeorgeon/Developmental-AI-Lab/blob/master/docs/agent3.ipynb)

# THE AGENT WHO TAMED THE TURTLE

# Learning objectives

Upon completing this lab, you will be able to assign appropriate valences to interactions, enabling a developmental agent to exhibit exploratory behavior in a simulated environment.

# Setup
## Import the turtle environment

In [2]:
from ColabTurtle.Turtle import *

## Define the Agent class

In [14]:
class Agent:
    def __init__(self, _valences):
        """ Creating our agent """
        self._valences = _valences
        self._action = None
        self._predicted_outcome = None

    def action(self, _outcome):
        """ tracing the previous cycle """
        if self._action is not None:
            print(f"Action: {self._action}, Prediction: {self._predicted_outcome}, Outcome: {_outcome}, " 
                  f"Prediction: {self._predicted_outcome == _outcome}, Valence: {self._valences[self._action][_outcome]}")

        """ Computing the next action to enact """
        # # TODO: Implement the agent's decision mechanism
        # self._action = 0
        # # TODO: Implement the agent's anticipation mechanism
        # self._predicted_outcome = 0
        return self._action


## Define the tutle environment class

You don't need to worry about the code of the ColabTurtleEnvironment below.

Just know that this environment:
* interprets the agent's actions as follows  `0`: move forward, `1`: turn left, `2`: turn right.
* returns outcome `1` when the turtle bumps into the border of the window, and `0` otherwise.

In [15]:
# @title Initialize the turtle environment

BORDER_WIDTH = 20

class ColabTurtleEnvironment:

    def __init__(self):
        """ Creating the Turtle window """
        bgcolor("lightGray")
        penup()
        goto(window_width() / 2, window_height()/2)
        face(0)
        pendown()
        color("green")

    def outcome(self, action):
        """ Enacting an action and returning the outcome """
        _outcome = 0
        for i in range(10):
            # _outcome = 0
            if action == 0:
                # move forward
                forward(10)
            elif action == 1:
                # rotate left
                left(4)
                forward(2)
            elif action == 2:
                # rotate right
                right(4)
                forward(2)

            # Bump on screen edge and return outcome 1
            if xcor() < BORDER_WIDTH:
                goto(BORDER_WIDTH, ycor())
                _outcome = 1
            if xcor() > window_width() - BORDER_WIDTH:
                goto(window_width() - BORDER_WIDTH, ycor())
                _outcome = 1
            if ycor() < BORDER_WIDTH:
                goto(xcor(), BORDER_WIDTH)
                _outcome = 1
            if ycor() > window_height() - BORDER_WIDTH:
                goto(xcor(), window_height() -BORDER_WIDTH)
                _outcome = 1

            # Change color
            if _outcome == 0:
                color("green")
            else:
                # Finit l'interaction
                color("red")
                # if action == 0:
                #     break
                if action == 1:
                    for j in range(10):
                        left(4)
                elif action == 2:
                    for j in range(10):
                        right(4)
                break

        return _outcome

## Define the valence of interactions

In [16]:
valences = [[1, -1], 
            [-1, -1],
            [-1, -1]]

The valence table specifies the valence of each interaction. An interaction is a tuple (action, outcome):

|| 0 Not bump | 1 Bump|
|---|---|---|
| 0 Forward| 1 | -1 |
| 1 Left | -1 | -1 |
| 2 Right| -1 | -1 |

## Instantiate the agent

In [17]:
a = Agent(valences)

## Run the simulation 

In [18]:
# @title Run the simulation

initializeTurtle()

# Parameterize the rendering
bgcolor("lightGray")
penup()
goto(window_width() / 2, window_height()/2)
face(0)
pendown()
color("green")
speed(10)

e = ColabTurtleEnvironment()

outcome = 0
for i in range(10):
    action = a.action(outcome)
    outcome = e.outcome(action)

Observe the turtle moving in a straigt line until it bumps into the border of the window

# PRELIMINARY EXERCISE

Copy Agent2 that you designed in your previous assignment to this notebook. 

Observe how your Agent2 behaves in this environment 

# ASSIGNMENT

Implement Agent3 by modifying your previous Agent2 such that it can select 3 possible actions: `0`, `1`, or `2`.

Choose the valences of interactions so that the agent does not remain stuck in a corner of the environment. 

## Create Agent3 by overriding the class Agent or your previous class Agent2

In [24]:
class Agent3(Agent):
    def __init__(self, _valences, possible_action = [0,1], min_count_memorize=0, nb_boring=4):
        super().__init__(_valences)
        self.min_count_memorize = min_count_memorize
        self.nb_boring = nb_boring
        self._possible_action = possible_action
        self._action_index = 0
        self._action = 0
        self._streak = 0
        
        self._memory = {}

    def action(self, _outcome):
        super().action(_outcome)

        self.update_memory(self._action, _outcome)

        new_action, self._predicted_outcome = self.best_action_and_prediction()

        if new_action == self._action:
            self._streak += 1
        else:
            self._streak = 0

        if self._streak >= self.nb_boring:
            
            new_action = self.next_action()
            if new_action == self._action:
                new_action = self.next_action()
            self._streak = 0
            
        self._action = new_action

        return self._action

    def update_memory(self, action, outcom):
        key = (action, outcom)
        if key not in self._memory:
            self._memory[key] = {"count": 0, "valence" : self._valences[action][outcom]}
        else:
            old_instance = self._memory[key]
            self._memory[key] = {"count": old_instance["count"], "valence" : self._valences[action][outcom]}
        entry = self._memory[key]
        entry["count"] += 1
        
    def best_action_and_prediction(self) -> tuple[int, int]:
        interactions = []
        for action in self._possible_action:
            outcome = self.get_most_probable_outcome(action)
            key = (action, outcome)
            entry = self._memory.get(key)
            if entry is not None:
                interactions.append(
                    {
                        "action": action, 
                        "outcome": outcome,
                        "valence": entry["valence"],
                        "count": entry["count"]
                    }
                )
        best = max(interactions, key=lambda x:x["valence"])
        return best["action"], best["outcome"]

    def next_action(self):
        self._action_index = (self._action_index + 1) % len(self._possible_action)
        return self._possible_action[self._action_index]

    def get_most_probable_outcome(self, action, min_count=0):

        candidates = [
            (entry["count"], outcome)
            for (mem_action, outcome), entry in self._memory.items()
            if mem_action == action and entry["count"] >= min_count
        ]
        return max(candidates, key=lambda x: x)[1] if candidates else 0

## Choose the valence table

Replace the `valences` table by your choice in the code below

In [25]:
valences = [[1, -5], 
            [-1, -2],
            [-1, -2]]

## Test your agent in the TurtleEnvironment

In [26]:
initializeTurtle()

# Parameterize the rendering
bgcolor("lightGray")
penup()
goto(window_width() / 2, window_height()/2)
face(0)
pendown()
color("green")
speed(10)

a = Agent3(valences, possible_action = [0, 1, 2])
e = ColabTurtleEnvironment()

outcome = 0
for i in range(100):
    action = a.action(outcome)
    outcome = e.outcome(action)

Action: 0, Prediction: None, Outcome: 0, Prediction: False, Valence: 1
Action: 0, Prediction: 0, Outcome: 0, Prediction: True, Valence: 1
Action: 0, Prediction: 0, Outcome: 0, Prediction: True, Valence: 1
Action: 0, Prediction: 0, Outcome: 0, Prediction: True, Valence: 1
Action: 1, Prediction: 0, Outcome: 0, Prediction: True, Valence: -1
Action: 0, Prediction: 0, Outcome: 1, Prediction: False, Valence: -5
Action: 0, Prediction: 0, Outcome: 1, Prediction: False, Valence: -5
Action: 0, Prediction: 0, Outcome: 1, Prediction: False, Valence: -5
Action: 0, Prediction: 0, Outcome: 1, Prediction: False, Valence: -5
Action: 1, Prediction: 0, Outcome: 1, Prediction: False, Valence: -2
Action: 1, Prediction: 1, Outcome: 1, Prediction: True, Valence: -2
Action: 1, Prediction: 1, Outcome: 0, Prediction: False, Valence: -1
Action: 1, Prediction: 1, Outcome: 0, Prediction: False, Valence: -1
Action: 2, Prediction: 0, Outcome: 0, Prediction: True, Valence: -1
Action: 1, Prediction: 0, Outcome: 0, Pre

## Improve your agent's code

If your agent gets stuck against a border or in a corner, modify the valences or the code. 
Try different ways to handle boredome or to select random actions. 
In the next lab, you will see how to design an agent that can adapt to the context. 

## Report 

Explain what you programmed and what results you observed. Export this document as PDF including your code, the traces you obtained, and your explanations below (no more than a few paragraphs):